# Molecular dynamics & free-energy analysis

Post-processing of GROMACS simulations: plotting time-series observables from .xvg output, building protein–RNA distance/contact maps, visualising pairwise free-energy matrices, checking BAR free-energy convergence, and setting up chained EDS alchemical topologies with the SMArt library. Each section is an independent analysis; the EDS section requires the SMArt environment. Example topology inputs are in data/topology_files/.

**Contents**

1. [GROMACS .xvg time-series plots](#sec1)
2. [Protein-RNA distance matrix](#sec2)
3. [Simulation matrix heatmap](#sec3)
4. [BAR free-energy convergence](#sec4)
5. [EDS alchemical topology conversion](#sec5)

> Notebooks were consolidated from separate working scripts; unused and broken scripts were removed. Each section keeps its own imports and reads independently. Cell outputs were cleared, and input data files are referenced by generic names (not included).


<a id="sec1"></a>
## 1. GROMACS .xvg time-series plots

Loads GROMACS .xvg output files and plots simulation observables over time with consistent styling.

<sub>Source script: <code>md_xvg_timeseries.ipynb</code></sub>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Load XVG files
# --------------------------------------------------
def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            parts = line.split()
            if parts:
                data.append([float(x) for x in parts])
    return np.array(data)


# --------------------------------------------------
# 2. Apply Grace-like matplotlib style
# --------------------------------------------------
def use_grace_style(
    font_family="Arial",
    base_fontsize=12,
    label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=11,
    axes_linewidth=1.5,
    tick_length=6,
    tick_width=1.3,
    boxed=False
):
    plt.rcParams.update({
        "font.family": font_family,
        "font.size": base_fontsize,
        "axes.labelsize": label_fontsize,
        "axes.titlesize": label_fontsize,
        "xtick.labelsize": tick_fontsize,
        "ytick.labelsize": tick_fontsize,
        "legend.fontsize": legend_fontsize,
        "axes.linewidth": axes_linewidth,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": tick_length,
        "ytick.major.size": tick_length,
        "xtick.major.width": tick_width,
        "ytick.major.width": tick_width,
        "legend.frameon": False,
        "axes.grid": False,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.bbox": "tight"
    })

    return {"boxed": boxed, "axes_linewidth": axes_linewidth}


# --------------------------------------------------
# 3. Generic Grace-like line plotter
# --------------------------------------------------
def plot_xvg_series(
    files_to_plot,
    xlabel="Time (ns)",
    ylabel="Value",
    title=None,
    figsize=(6, 4.5),
    colors=None,
    linewidth=2,
    convert_time_ps_to_ns=True,
    xlim=None,
    ylim=None,
    legend_loc="upper right",
    boxed=False,
    savepath=None
):
    fig, ax = plt.subplots(figsize=figsize)

    for label, fname in files_to_plot.items():
        data = load_xvg(fname)

        x = data[:, 0]
        y = data[:, 1]

        if convert_time_ps_to_ns:
            x = x / 1000.0

        color = None if colors is None else colors.get(label, None)

        ax.plot(x, y, label=label, linewidth=linewidth, color=color)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    if title is not None:
        ax.set_title(title)

    if not boxed:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    else:
        for spine in ax.spines.values():
            spine.set_linewidth(1.5)

    ax.tick_params(direction="out", length=6, width=1.3)

    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.legend(loc=legend_loc)

    plt.tight_layout()

    if savepath is not None:
        plt.savefig(savepath, dpi=300)

    plt.show()

    return fig, ax

In [ ]:
use_grace_style(boxed=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply style once
use_grace_style(
    font_family="Arial",
    base_fontsize=12,
    label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=11,
    boxed=False
)

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

files = {
    "Complex": "6gbm_RRM_whole_complex_rms.xvg",
    "RNA": "6gbm_RRM_whole_RNA_rms.xvg",
    "Protein": "6gbm_RRM_whole_protein_rms.xvg"
}

colors = {
    "Complex": "black",
    "RNA": "tab:blue",
    "Protein": "tab:green"
}

fig, ax = plt.subplots(figsize=(6, 4.5))   # 4:3 ratio

for label, fname in files.items():
    data = load_xvg(fname)

    time_ns = data[:, 0] / 1000.0   # convert ps -> ns
    rmsd = data[:, 1]

    ax.plot(time_ns, rmsd, label=label, linewidth=2, color=colors[label])

ax.set_xlabel("Time (ns)")
ax.set_ylabel("RMSD (nm)")

# Grace-like axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=6, width=1.3)

ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.savefig("RMSD_6gbm_whole_xmgrace_style.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply style once
use_grace_style(
    font_family="Arial",
    base_fontsize=12,
    label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=11,
    boxed=False
)

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

files = {
    "Complex": "6snj_RRM_whole_complex_rms.xvg",
    "RNA": "6snj_RRM_whole_RNA_rms.xvg",
    "Protein": "6snj_RRM_whole_protein_rms.xvg"
}

fig, ax = plt.subplots(figsize=(6, 4.5))   # 4:3 ratio

for label, fname in files.items():
    data = load_xvg(fname)

    time_ns = data[:, 0] / 1000.0   # convert ps -> ns
    rmsd = data[:, 1]

    ax.plot(time_ns, rmsd, label=label, linewidth=2, color=colors[label])

ax.set_xlabel("Time (ns)")
ax.set_ylabel("RMSD (nm)")

# Grace-like axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=6, width=1.3)

ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.savefig("6snj RRM_whole_RMSD_xmgracestyle.png")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

files = {
    "Complex": "6g99_ZnF_UGGUG_complex_rms.xvg",
    "RNA": "6g99_ZnF_UGGUG_RNA_rms.xvg",
    "Protein": "6g99_ZnF_UGGUG_protein_rms.xvg"
}

plt.figure(figsize=(8,5))

for label, fname in files.items():
    data = load_xvg(fname)
    plt.plot(data[:,0], data[:,1], label=label, linewidth=2)

plt.xlabel("Time (ps)")
plt.ylabel("RMSD (nm)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply style once
use_grace_style(
    font_family="Arial",
    base_fontsize=12,
    label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=11,
    boxed=False
)

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

files = {
    "Complex": "6g99_ZnF_UGGUG_complex_rms.xvg",
    "RNA": "6g99_ZnF_UGGUG_RNA_rms.xvg",
    "Protein": "6g99_ZnF_UGGUG_protein_rms.xvg"
}

colors = {
    "Complex": "black",
    "RNA": "tab:blue",
    "Protein": "tab:green"
}

fig, ax = plt.subplots(figsize=(6, 4.5))   # 4:3 ratio

for label, fname in files.items():
    data = load_xvg(fname)

    time_ns = data[:, 0] / 1000.0   # convert ps -> ns
    rmsd = data[:, 1]

    ax.plot(time_ns, rmsd, label=label, linewidth=2, color=colors[label])

ax.set_xlabel("Time (ns)")
ax.set_ylabel("RMSD (nm)")

# Grace-like axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=6, width=1.3)

ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.savefig("RMSD_6g99_ZnF_xmgrace_style.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

files = {
    "Complex": "6gbm_RRM_stemloop_complex_md_100ns_complex.xvg",
    "RNA": "6gbm_RRM_stemloop_complex_md_100ns_RNA.xvg",
    "Protein": "6gbm_RRM_stemloop_complex_md_100ns_protein.xvg"
}

plt.figure(figsize=(8,5))

for label, fname in files.items():
    data = load_xvg(fname)
    plt.plot(data[:,0], data[:,1], label=label, linewidth=2)

plt.xlabel("Time (ps)")
plt.ylabel("RMSD (nm)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply style once
use_grace_style(
    font_family="Arial",
    base_fontsize=12,
    label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=11,
    boxed=False
)

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

files = {
    "Complex": "6gbm_complex_md_100ns_RMSD_complex.xvg",
    "RNA": "6gbm_complex_md_100ns_RMSD_RNA.xvg",
    "Protein": "6gbm_complex_md_100ns_RMSD_protein.xvg"
}

colors = {
    "Complex": "black",
    "RNA": "tab:blue",
    "Protein": "tab:green"
}

fig, ax = plt.subplots(figsize=(6, 4.5))   # 4:3 ratio

for label, fname in files.items():
    data = load_xvg(fname)

    time_ns = data[:, 0] / 1000.0   # convert ps -> ns
    rmsd = data[:, 1]

    ax.plot(time_ns, rmsd, label=label, linewidth=2, color=colors[label])

ax.set_xlabel("Time (ns)")
ax.set_ylabel("RMSD (nm)")

# Grace-like axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=6, width=1.3)

ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.savefig("RMSD_6gbm_RRM_cut_xmgrace_style.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

files = {
    "Complex": "6snj_complex_md_100ns_rms_complex.xvg",
    "RNA": "6snj_complex_md_100ns_rms_RNA.xvg",
    "Protein": "6snj_complex_md_100ns_rms_protein.xvg"
}

plt.figure(figsize=(8,5))

for label, fname in files.items():
    data = load_xvg(fname)
    plt.plot(data[:,0], data[:,1], label=label, linewidth=2)

plt.xlabel("Time (ps)")
plt.ylabel("RMSD (nm)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply style once
use_grace_style(
    font_family="Arial",
    base_fontsize=12,
    label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=11,
    boxed=False
)

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

files = {
    "Complex": "6snj_complex_md_100ns_rms_complex.xvg",
    "RNA": "6snj_complex_md_100ns_rms_RNA.xvg",
    "Protein": "6snj_complex_md_100ns_rms_protein.xvg"
}

colors = {
    "Complex": "black",
    "RNA": "tab:blue",
    "Protein": "tab:green"
}

fig, ax = plt.subplots(figsize=(6, 4.5))   # 4:3 ratio

for label, fname in files.items():
    data = load_xvg(fname)

    time_ns = data[:, 0] / 1000.0   # convert ps -> ns
    rmsd = data[:, 1]

    ax.plot(time_ns, rmsd, label=label, linewidth=2, color=colors[label])

ax.set_xlabel("Time (ns)")
ax.set_ylabel("RMSD (nm)")

# Grace-like axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=6, width=1.3)

ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.savefig("RMSD_6snj_RRM_cut_xmgrace_style.png")
plt.show()

In [ ]:
from matplotlib import colormaps
list(colormaps)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

# Define all files to plot
files_to_plot = {
    "RMSD (Complex)": "rmsd_complex_EDS_systemequilibration.xvg",
    "RMSD (RNA)": "rmsd_RNA.xvg",
    "RMSD (Protein)": "rmsd_protein.xvg"
}

plt.figure(figsize=(8,5))

for label, fname in files_to_plot.items():
    data = load_xvg(fname)
    plt.plot(data[:,0], data[:,1], label=label, linewidth=2)

plt.xlabel("Time (ps)")
plt.ylabel("RMSD (nm)")
plt.title("RMSD NPT and NVT equilibration of EDS containing structure")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

# ---- xmgrace-like styling ----
plt.rcParams.update({
    "font.family": "arial",
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 1.5,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.3,
    "ytick.major.width": 1.3,
    "legend.frameon": False
})

# Define all files to plot
files_to_plot = {
    "Complex": "rmsd_complex_EDS_systemequilibration.xvg",
    "RNA": "rmsd_RNA.xvg",
    "Protein": "rmsd_protein.xvg"
}

# Optional fixed colors for cleaner xmgrace-like appearance
colors = {
    "Complex": "black",
    "RNA": "tab:blue",
    "Protein": "tab:green"
}

fig, ax = plt.subplots(figsize=(6, 4.5))  # 4:3 ratio

for label, fname in files_to_plot.items():
    data = load_xvg(fname)
    time_ns = data[:, 0] / 1000.0   # convert ps -> ns
    ax.plot(time_ns, data[:, 1], label=label, linewidth=2, color=colors[label])

ax.set_xlabel("Time (ns)")
ax.set_ylabel("RMSD (nm)")
# ax.set_title("RMSD during equilibration")  # optional; often better in caption

# xmgrace-like axis appearance
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(loc="lower right")

plt.tight_layout()
plt.savefig("equilibration_stability_ZnF.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Grace-like style (same as your RMSD plots)
def use_grace_style():
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 12,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11,
        "axes.linewidth": 1.5,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 6,
        "ytick.major.width": 1.3,
        "ytick.major.size": 6,
        "ytick.major.width": 1.3,
        "legend.frameon": False,
        "axes.grid": False
    })

use_grace_style()

# --------------------------------------------------
# XVG loader
# --------------------------------------------------

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

# --------------------------------------------------
# Load RMSF data
# --------------------------------------------------

data = load_xvg("6g99_ZnF_UGGUG_RNA_rmsf.xvg")

residue = data[:,0]
rmsf = data[:,1]

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(6,4.5))   # 4:3 thesis ratio

ax.plot(residue, rmsf,
        color="black",
        linewidth=2)

ax.set_xlabel("RNA residue")
ax.set_ylabel("RMSF (nm)")

# Grace-like axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=6, width=1.3)

# Set x-axis ticks to show only whole numbers
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("RMSF_6g99_ZnF_xmgrace_style.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Grace-style settings
# -----------------------------

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "axes.linewidth": 1.5,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 6,
    "ytick.major.width": 1.3,
    "ytick.major.size": 6,
    "ytick.major.width": 1.3,
    "legend.frameon": False
})

# -----------------------------
# XVG loader
# -----------------------------

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

# -----------------------------
# Load data
# -----------------------------

data = load_xvg("6g99_ZnF_UGGUG_md_cont_num.xvg")

time_ns = data[:,0] / 1000
contacts = data[:,1]

# -----------------------------
# Plot
# -----------------------------

fig, ax = plt.subplots(figsize=(6,4.5))

ax.plot(time_ns, contacts,
        color="black",
        linewidth=2)

ax.set_xlabel("Time (ns)")
ax.set_ylabel("Number of contacts")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(direction="out", length=6, width=1.3)

plt.tight_layout()
plt.savefig("contacts_6g99_ZnF_xmgrace_style.png")
plt.show()

In [ ]:
files = {
    "U1": "6g99_ZnF_UGGUG_md_cont_num_U1.xvg",
    "G2": "6g99_ZnF_UGGUG_md_cont_num_G2.xvg",
    "U3": "6g99_ZnF_UGGUG_md_cont_num_U3.xvg",
    "U4": "6g99_ZnF_UGGUG_md_cont_num_U4.xvg",
    "G5": "6g99_ZnF_UGGUG_md_cont_num_G5.xvg"
}

colors = {
    "U1": "tab:blue",
    "G2": "tab:green",
    "U3": "tab:orange",
    "U4": "tab:red",
    "G5": "black"
}

fig, ax = plt.subplots(figsize=(6,4.5))

for label, fname in files.items():

    data = load_xvg(fname)

    time_ns = data[:,0] / 1000
    contacts = data[:,1]

    ax.plot(time_ns,
            contacts,
            label=label,
            linewidth=0.5,
            color=colors[label])

ax.set_xlabel("Time (ns)")
ax.set_ylabel("Number of contacts")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(direction="out", length=6, width=1.3)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Grace-like style (same as your RMSD plots)
def use_grace_style():
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 12,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11,
        "axes.linewidth": 1.5,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 6,
        "ytick.major.width": 1.3,
        "ytick.major.size": 6,
        "ytick.major.width": 1.3,
        "legend.frameon": False,
        "axes.grid": False
    })

use_grace_style()

# --------------------------------------------------
# XVG loader
# --------------------------------------------------

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

# --------------------------------------------------
# Load RMSF data
# --------------------------------------------------

data = load_xvg("6gbm_RRM_whole_md_RNA_rmsf.xvg")

residue = data[:,0]
rmsf = data[:,1]

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(6,4.5))   # 4:3 thesis ratio

ax.plot(residue, rmsf,
        color="black",
        linewidth=2)

ax.set_xlabel("RNA residue")
ax.set_ylabel("RMSF (nm)")

# Grace-like axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=6, width=1.3)

# Set x-axis ticks to show only whole numbers
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("RMSF_6gbm_RRM_xmgrace_style.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Grace-like style (same as your RMSD plots)
def use_grace_style():
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 12,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11,
        "axes.linewidth": 1.5,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 6,
        "ytick.major.width": 1.3,
        "ytick.major.size": 6,
        "ytick.major.width": 1.3,
        "legend.frameon": False,
        "axes.grid": False
    })

use_grace_style()

# --------------------------------------------------
# XVG loader
# --------------------------------------------------

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

# --------------------------------------------------
# Load RMSF data
# --------------------------------------------------

data = load_xvg("6snj_RRM_whole_RNA_rmsf.xvg")

residue = data[:,0]
rmsf = data[:,1]

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(6,4.5))   # 4:3 thesis ratio

ax.plot(residue, rmsf,
        color="black",
        linewidth=2)

ax.set_xlabel("RNA residue")
ax.set_ylabel("RMSF (nm)")

# Grace-like axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=6, width=1.3)

# Set x-axis ticks to show only whole numbers
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("RMSF_6snj_RRM_xmgrace_style.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# XVG loader
# --------------------------------------------------

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

# --------------------------------------------------
# Load Rg data
# --------------------------------------------------

data = load_xvg("6gbm_complex_md_100ns_gyr.xvg")

time_ns = data[:,0] / 1000
rg = data[:,1]

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(6,4.5))

ax.plot(time_ns,
        rg,
        color="black",
        linewidth=2)

ax.set_xlabel("Time (ns)")
ax.set_ylabel("R$_g$ (nm)")

# Grace-style axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(direction="out", length=6, width=1.3)

plt.tight_layout()
plt.savefig("gyr_6gbm_complex_xmgrace_style.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# XVG loader
# --------------------------------------------------

def load_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(('#','@')):
                continue
            data.append([float(x) for x in line.split()])
    return np.array(data)

# --------------------------------------------------
# Load Rg data
# --------------------------------------------------

data = load_xvg("6snj_complex_md_100ns_gyr.xvg")

time_ns = data[:,0] / 1000
rg = data[:,1]

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(6,4.5))

ax.plot(time_ns,
        rg,
        color="black",
        linewidth=2)

ax.set_xlabel("Time (ns)")
ax.set_ylabel("R$_g$ (nm)")

# Grace-style axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(direction="out", length=6, width=1.3)

plt.tight_layout()
plt.savefig("gyr_6snj_complex_xmgrace_style.png")
plt.show()

<a id="sec2"></a>
## 2. Protein-RNA distance matrix

Reads a GROMACS distance-matrix .xvg for a protein-RNA system and renders it as a contact/distance map.

<sub>Source script: <code>protein_rna_distance_matrix.ipynb</code></sub>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#path of the data file
path = "dist_matrix_prot_RNA.xvg"


In [ ]:
X = []
with open(path, "r") as file:
    for i, line in enumerate(file):
        if line.startswith(("@", "#")):
            continue

        # Split the line and add to respective lists
        line = line.strip()
        line = line.split('   ')
        values = [float(item) for item in line]
        rowlength = len(values)-1

        X.append(values[1:])
X = np.mean(X, axis=0)
print(rowlength)

In [ ]:
    plt.figure(figsize=(8, 6))
    plt.imshow(X.reshape(23, -1), cmap='viridis', aspect='auto')
    plt.colorbar(label='Value')
    plt.xlabel("Columns")
    plt.ylabel("Rows")
    # Save the heatmap to the specified path
    output_path = f"dist_matrix_prot_RNA.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()


In [ ]:
    # Save the heatmap to the specified path
    output_path = f"dist_matrix_prot_RNA.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()


<a id="sec3"></a>
## 3. Simulation matrix heatmap

Turns a pairwise simulation matrix (e.g. relative free energies / distances between states) into an annotated heatmap.

<sub>Source script: <code>simulation_matrix_heatmap.ipynb</code></sub>

In [ ]:
array([[  0.        ,  37.73972126,  38.57929403,  37.32868976],
       [-37.73972126,   0.        ,   0.83957277,  -0.4110315 ],
       [-38.57929403,  -0.83957277,   0.        ,  -1.25060428],
       [-37.32868976,   0.4110315 ,   1.25060428,   0.        ]])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# The array from the previous cell
data_array = np.array([[  0.        ,  37.73972126,  38.57929403,  37.32868976],
       [-37.73972126,   0.        ,   0.83957277,  -0.4110315 ],
       [-38.57929403,  -0.83957277,   0.        ,  -1.25060428],
       [-37.32868976,   0.4110315 ,   1.25060428,   0.        ]])

plt.figure(figsize=(8, 6))
ax = sns.heatmap(data_array, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('ddG with all 21 lambda poins, complex - RNA solution')
# Add 'ddG' as the label for the color bar
cbar = ax.collections[0].colorbar
cbar.set_label('ddG', fontsize=16)
plt.savefig("ddG_complex_RNA_11_lambda.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# The array from the previous cell
data_array_v2 = np.array([[  0.        ,  26.64197327,  23.89275236,  38.96694757],
       [-26.64197327,   0.        ,  -2.74922091,  12.3249743 ],
       [-23.89275236,   2.74922091,   0.        ,  15.0741952 ],
       [-38.96694757, -12.3249743 , -15.0741952 ,   0.        ]])

plt.figure(figsize=(8, 6))
ax = sns.heatmap(data_array_v2, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
cbar = ax.collections[0].colorbar
cbar.set_label('ddG')

plt.title('ddG with all 3 lambda poins, complex - RNA solution')
plt.savefig("ddG_complex_RNA_3_lambda.png")
plt.show()

In [ ]:
import pandas as pd

#comparison of sampling only each perturbation, vs sampling all the simulations

# Assuming symmetrical_matrix is already defined from the previous cell
# If not, you might need to re-run the previous cell or redefine it:
# symmetrical_matrix = np.array([[
#    0.        , -2.124274  ,  4.658745  , -3.16638   ],
#   [ 2.124274  ,  0.        ,  0.780548  , -1.382296  ],
#   [-4.658745  , -0.780548  ,  0.        ,  2.641434  ],
#   [ 3.16638   ,  1.382296  , -2.641434  ,  0.        ]])

df_symmetrical = pd.DataFrame(symmetrical_matrix)
display(df_symmetrical)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'symmetrical_matrix' is already defined in the kernel from previous execution

plt.figure(figsize=(8, 6))
sns.heatmap(symmetrical_matrix, annot=True, cmap='Greys', fmt=".3f", linewidths=.5)
plt.title('ddG of only using the perturbation for sampling and sampling all the simulations')
plt.xlabel('Starting state')
plt.ylabel('Ending state')
plt.show()

In [ ]:
import pandas as pd
import io

#Initial RNA simulations in water

# Raw data from the previous cell content
data_str = """
0 1 229.85 -205.68
0 2 -103.04 139.08
0 3 298.08 -290.28
1 2 -334.27 357.72
1 3 78.17 -75.31
2 3 429.26 -424.46
"""

# Read the data into a pandas DataFrame
df = pd.read_csv(io.StringIO(data_str), sep=' ', header=None, names=['Initial state', 'Final state', 'dG_forward|kJ/mol', 'dG_reverse|kJ/mol'])

display(df)

<a id="sec4"></a>
## 4. BAR free-energy convergence

Plots Bennett Acceptance Ratio (BAR) free-energy estimates across replicates to assess convergence of alchemical calculations.

<sub>Source script: <code>bar_free_energy_convergence.ipynb</code></sub>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

file_list = ["PTP_0_1_1_bar.xvg", "PTP_0_1_2_bar.xvg", "PTP_0_1_3_bar.xvg"]

for file_name in file_list:
    data = np.loadtxt(file_name, comments=('#','@'))
    plt.plot(data[:,0], data[:,1], label=file_name)

plt.xlabel("Time (ps)")
plt.ylabel("BAR (kJ/mol)")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import re

# Find all .xvg files in the current directory
file_list = glob.glob("*.xvg")

# Define the patterns to group by
patterns = ["PTP_0_1", "PTP_1_2", "PTP_2_3", "PTP_3_0"]

# Group files by pattern
grouped_files = {pattern: [] for pattern in patterns}
for file_name in file_list:
    for pattern in patterns:
        if re.search(pattern, file_name):
            grouped_files[pattern].append(file_name)
            break # Assume each file matches only one pattern

# Plot each group
for pattern, files in grouped_files.items():
    if files: # Only plot if there are files in the group
        plt.figure() # Create a new figure for each group
        for file_name in files:
            try:
                data = np.loadtxt(file_name, comments=('#','@'))
                plt.plot(data[:,0], data[:,1], label=file_name)
            except Exception as e:
                print(f"Could not process file {file_name}: {e}")

        plt.xlabel("Time (ps)")
        plt.ylabel("BAR (kJ/mol)")
        plt.title(f"Plots for {pattern}")
        plt.legend()
        plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt("PTP_0_1_2_bar.xvg", comments=('#','@'))
plt.plot(data[:,0], data[:,1])
plt.xlabel("Time (ps)")
plt.ylabel("BAR (kJ/mol)")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import re

# Find all .xvg files in the current directory
file_list = glob.glob("*.xvg")

# Define the patterns to group by in the desired order
patterns = ["PTP_0_1", "PTP_1_2", "PTP_2_3", "PTP_3_0"]

# Group files by pattern
grouped_files = {pattern: [] for pattern in patterns}
for file_name in file_list:
    for pattern in patterns:
        if re.search(pattern, file_name):
            grouped_files[pattern].append(file_name)
            break # Assume each file matches only one pattern

# Concatenate data from each group and then sequentially, adjusting time
combined_data = []
legend_labels = []

for pattern in patterns:
    files = grouped_files.get(pattern, [])
    if files: # Only process if there are files in the group
        group_data = []
        for file_name in files:
            try:
                data = np.loadtxt(file_name, comments=('#','@'))
                # Adjust time to start from 0 for each segment
                adjusted_data = data.copy()
                adjusted_data[:, 0] -= adjusted_data[0, 0] # Subtract the initial time
                group_data.append(adjusted_data)
            except Exception as e:
                print(f"Could not process file {file_name}: {e}")

        if group_data:
            # Concatenate data within the group (if multiple files per group)
            concatenated_group_data = np.concatenate(group_data) # This concatenates all files in the group
            combined_data.append(concatenated_group_data)
            legend_labels.append(pattern) # Add pattern to legend labels


# Create a single figure for all plots
plt.figure()

# Plot each concatenated group data sequentially
for i, data in enumerate(combined_data):
    plt.plot(data[:,0], data[:,1], label=legend_labels[i])

plt.xlabel("Time relative to segment start (ps)")
plt.ylabel("BAR (kJ/mol)")
plt.title("Thermodynamic Circle (Time Relative to Segment Start)")
plt.legend()
plt.ylim(bottom=0)  # Set the lower limit of the y-axis to 0
plt.show()

<a id="sec5"></a>
## 5. EDS alchemical topology conversion

Uses the SMArt library to build a maximum-common-structure and generate chained enveloping-distribution-sampling (EDS) topologies from GROMACS/GROMOS inputs.

<sub>Source script: <code>eds_topology_conversion.ipynb</code></sub>

In [ ]:
# import statement
import sys
######### clone smart and then add path to it here
#path_to_smart = '/../..path/to/SMArt'
path_to_smart = 'path/to/SMArt'
##########
sys.path.insert(0, path_to_smart)

import os
import SMArt
import pandas
import numpy
import scipy

from SMArt import alchemy
from SMArt.md import parse_top

Convert GROMACS topologies to GROMOS topologies(?)

In [ ]:
from setuptools import setup, find_packages

setup(
    name='SMArt',
    version='1.0.0',
    packages=find_packages(),
    license='GPLv3.0',
    author='Drazen Petrov',
    author_email='drazen.petrov@boku.ac.at',
    description='Simulation and Modelling Art',
    include_package_data=True,
)

## generate max common structure

In [ ]:
#Set the directory
%cd path/to/SMArt'topol_WT.top','topol_mutA.top','topol_mutC.top','topol_mutU.top']

TOPs = []
MTs=[]
for tf in top_files:
    top = parse_top(tf, format_type='gm', allow_multiple_matches=True)
    TOPs.append(top)
    MTs.append(top.molecules[0].mol_type)

In [ ]:
# generage the MCS
mcs = alchemy.get_EDS(*MTs, flag_get_core_common_atoms_top0=True)

sol_EDS, state_names = alchemy.generate_EDS_top(mcs, ff_dumm=TOPs[0].get_DUM_type)
sol_EDS

## generate topologies

In [ ]:
# check that we don't have any excl/pair perturbation
for atoms, ep in sol_EDS.toptp.excl_pair.items():
    temp_states = set()
    for st in ep.states:
        if st:
            temp_states.add(st.p)
    assert len(temp_states)==1
# this should pass, but it's still an extra check...

In [ ]:
# this is the general one (but it already contains state_1 to state_2 perturbation)
fd = "gmx_EDS_tops/"
if not os.path.isdir(fd):
    os.mkdir(fd)
sol_EDS.toptp.write_itp(fd+"/EDS.itp", flag_generate_excl_pairs = True)

In [ ]:
# now we create a chain of topologies where we go from one state to another
# 0_1 means perturbation between state 1 and 2 (LYS -> K3C in this case)
# 1_2 is state 2 to state 3 (K3C -> KAC in this case)
# and so on
# n_0 is the last state to the first one (KAC -> LYS in this case)

for top_i in range(len(sol_EDS.tops)):
    print(top_i)
    other_st = (top_i+1)%len(sol_EDS.tops)
    sol_EDS.toptp.set_top_state(top_i, other_state=other_st, find_other_state=1)
    sol_EDS.toptp._gm_generate_excl_pairs()
    for ep in sol_EDS.toptp.vdw_pairs:ep.states = ep.states*len(sol_EDS.tops)
    for ep in sol_EDS.toptp.exclusions:ep.states = ep.states*len(sol_EDS.tops)
    sol_EDS.toptp.write_itp(fd+'/EDS_{:}_{:}.itp'.format(top_i, other_st),
                            state=top_i, top_state=top_i,
                            other_state=other_st, find_other_state=1)